In [1]:
# Import all basic functions
import onnx
import onnxruntime as ort
import ml_dtypes
import numpy as np
import h5py
import torch
import torch.utils.dlpack

# Import config
from config import load_config, InputSourceConfig

import importlib
import sys
import tqdm

DEVICE = torch.device("cuda")


In [2]:
# data generation
from generate_data.datagen import (
    sample_ambient_points,
    sample_around_points_gpu,
    repeat_points_to_match,
    get_token_ids_array,
    knn_gpu,
    estimate_local_tangent_space_p_var,
)

# model generation
from construct.construct import (
    ensure_base_model,
    ensure_subgraph_pullback,
    ensure_subgraphs,
    ensure_extra_outputs,
)

# execution code
from execute.optimization_loop import optimization_loop, NewtonRaphson
from execute.io_onnx import *


In [ ]:
importlib.reload(sys.modules["execute.optimization_loop"])
importlib.reload(sys.modules["generate_data.datagen"])
importlib.reload(sys.modules["construct.construct"])
importlib.reload(sys.modules["execute.io_onnx"])

In [3]:
# Load config

cfg = load_config("configs/smoll.toml")
# makes sure that our base model is loaded
ensure_base_model(cfg, force=False, verbose=False)
extra_outputs = [sublayer.output for sublayer in cfg.sublayers]
model_path = ensure_extra_outputs(cfg, extra_outputs, force=True)


Base model already exists, skipping export -> onnx_models/smoll/base_model.onnx
Extra outputs model generated -> onnx_models/smoll/base_model_extra_outputs.onnx


In [4]:
# Load n_samples real sequences of length cfg.seq, then for every
# (sample, position) pair substitute a random subsample of the vocabulary
# into that position, holding the other cfg.seq-1 positions fixed at the
# original sequence's tokens.
n_samples = 100
vocab_sample_size = 1000  # per (sample, position); full vocab is cfg.vocab_size

rng = np.random.default_rng(0)

input_source = cfg.input_source
input_source.data_batch_size = n_samples
base_seqs = get_token_ids_array(input_source, seq_len=cfg.seq)  # (n_samples, cfg.seq)

# vocab_samples[sample_idx, pos] = vocab_sample_size token ids drawn for that slot
vocab_samples = rng.integers(
    0, cfg.vocab_size, size=(n_samples, cfg.seq, vocab_sample_size)
)

# perturbed[sample_idx, pos] has shape (vocab_sample_size, cfg.seq):
#   column `pos` = vocab_samples[sample_idx, pos], all other columns =
#   base_seqs[sample_idx]
perturbed = np.broadcast_to(
    base_seqs[:, None, None, :], (n_samples, cfg.seq, vocab_sample_size, cfg.seq)
).copy()
for pos in range(cfg.seq):
    perturbed[:, pos, :, pos] = vocab_samples[:, pos, :]

# flatten (sample, pos, vocab_sample) into rows, then chunk into
# (cfg.batch, cfg.seq) blocks for batched inference, matching the pattern
# used for `inputs` above.
perturbed_flat = perturbed.reshape(-1, cfg.seq)
n_rows = perturbed_flat.shape[0]
pad = (-n_rows) % cfg.batch
if pad:
    perturbed_flat = np.concatenate(
        [perturbed_flat, np.repeat(perturbed_flat[-1:], pad, axis=0)], axis=0
    )
perturbed_inputs = perturbed_flat.reshape(-1, cfg.batch, cfg.seq)

# per-row annotation: which sample, which position was swept, which vocab id
# was substituted in -- recovered via the same (sample, pos, vocab_sample)
# ordering used to build `perturbed` (padding rows just repeat the last row).
row_sample_idx, row_pos, row_vocab_sample_idx = np.unravel_index(
    np.arange(n_rows), (n_samples, cfg.seq, vocab_sample_size)
)
row_vocab_id = vocab_samples[row_sample_idx, row_pos, row_vocab_sample_idx]

print(perturbed_inputs.shape, "n_rows (unpadded):", n_rows)


2026-09-09 13:07:50,924 huggingface_hub.utils._http [WARNING] - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(25000, 32, 8) n_rows (unpadded): 800000


In [5]:

# Define output session
sess_options = ort.SessionOptions()

# Very Important --- will cause errors if missing
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_BASIC
sess_options.intra_op_num_threads = 8  # pick a value <= your usable core count

out_sess = ort.InferenceSession(
    str(model_path),
    sess_options=sess_options,
    providers=["CUDAExecutionProvider"],
)

# Renamed + ordered lists -- see note in the embedding_gen cell above.
out_outputs_meta = out_sess.get_outputs()
out_inputs_meta = out_sess.get_inputs()
out_output_names = [o.name for o in out_outputs_meta]
out_input_names = [o.name for o in out_inputs_meta]

print(out_output_names, out_input_names)

['output', 'add_6', 'add_8', 'add_12', 'add_14', 'add_18', 'add_20', 'add_24', 'add_26', 'add_30', 'add_32', 'add_36', 'add_38', 'add_42', 'add_44', 'add_48', 'add_50', 'add_54', 'add_56', 'add_60', 'add_62', 'add_66', 'add_68', 'add_72', 'add_74', 'add_78', 'add_80', 'add_84', 'add_86', 'add_90', 'add_92', 'add_96', 'add_98', 'add_102', 'add_104', 'add_108', 'add_110', 'add_114', 'add_116', 'add_120', 'add_122', 'add_126', 'add_128', 'add_132', 'add_134', 'add_138', 'add_140', 'add_144', 'add_146', 'add_150', 'add_152', 'add_156', 'add_158', 'add_162', 'add_164', 'add_168', 'add_170', 'add_174', 'add_176', 'add_180', 'add_182', 'add_186', 'add_188', 'add_192', 'add_194'] ['input_ids']


In [6]:
# Analytical size estimate (no enumeration of the ~64 layer groups): every
# layer group has the same per-row schema, so total size is just
# n_layers * n_rows * (bytes per row) plus the small fixed-size tables.
INT64 = 8
FLOAT32 = 4

n_layers = len(out_output_names) - 1

# perturbed_sweep.h5: per layer, per row -- vec (d_model float32) +
# sample_idx/pos/vocab_id (int64 each); plus one base_seqs (n_samples, seq) table.
sweep_bytes_per_row = cfg.d_model * FLOAT32 + 3 * INT64
sweep_size = n_layers * n_rows * sweep_bytes_per_row + n_samples * cfg.seq * INT64

# local_tangent_spaces.h5: per layer, per row -- tangent_basis (d_model^2
# float32) + k/sample_idx/pos/vocab_id (int64 each) + neighbor_vocab_ids
# ((d_model + 1) int64).
tangent_bytes_per_row = (
    cfg.d_model**2 * FLOAT32 + 4 * INT64 + (cfg.d_model + 1) * INT64
)
tangent_size = n_layers * n_rows * tangent_bytes_per_row

for name, size in [
    ("perturbed_sweep.h5", sweep_size),
    ("local_tangent_spaces.h5", tangent_size),
]:
    print(
        f"{name}: n_layers={n_layers} n_rows={n_rows:,} d_model={cfg.d_model} "
        f"-> {size / 2**30:,.3f} GiB ({size:,} bytes, uncompressed, ignoring "
        f"h5 header/metadata overhead)"
    )


perturbed_sweep.h5: n_layers=64 n_rows=800,000 d_model=960 -> 184.250 GiB (197,836,806,400 bytes, uncompressed, ignoring h5 header/metadata overhead)
local_tangent_spaces.h5: n_layers=64 n_rows=800,000 d_model=960 -> 176,149.368 GiB (189,138,944,000,000 bytes, uncompressed, ignoring h5 header/metadata overhead)


In [7]:
# Run the perturbed sweep through the model and save, per output layer, the
# d_model vector at the swept position only -- annotated with which sample,
# sequence position, and vocab id produced it. Streams straight to h5 (one
# output layer at a time) instead of buffering all layers x all rows in
# memory, since n_rows can be in the hundreds of thousands. Flushed every
# flush_every chunks so a crash/interrupt partway through still leaves a
# readable file with whatever rows made it in so far.
import os
perturbed_h5_path = "onnx_models/smoll/perturbed_sweep.h5"
os.makedirs(os.path.dirname(perturbed_h5_path), exist_ok=True)

n_chunks = perturbed_inputs.shape[0]
flush_every = 50  # chunks

with h5py.File(perturbed_h5_path, "w") as h5f:
    h5f.attrs["n_samples"] = n_samples
    h5f.attrs["vocab_sample_size"] = vocab_sample_size
    h5f.attrs["seq"] = cfg.seq
    h5f.attrs["d_model"] = cfg.d_model

    # original (unperturbed) sequences, indexed by sample_idx -- each row's
    # sample_idx (saved per layer below) points into this table, so the
    # original sequence a row was derived from is base_seqs[sample_idx].
    h5f.create_dataset("base_seqs", data=base_seqs)

    datasets = {}
    for output_name in out_output_names[1:]:
        grp = h5f.create_group(output_name)
        grp.attrs["output_name"] = output_name
        datasets[output_name] = {
            "vec": grp.create_dataset(
                "vec", shape=(n_rows, cfg.d_model), dtype="float32"
            ),
            "sample_idx": grp.create_dataset("sample_idx", shape=(n_rows,), dtype="int64"),
            "pos": grp.create_dataset("pos", shape=(n_rows,), dtype="int64"),
            "vocab_id": grp.create_dataset("vocab_id", shape=(n_rows,), dtype="int64"),
        }

    row_start = 0
    for c in tqdm.tqdm(range(n_chunks)):
        token_ids = perturbed_inputs[c]  # (cfg.batch, cfg.seq)
        fx = run_iobinding_cpu_input_gpu_output(
            out_sess,
            out_input_names[0],
            token_ids,
            out_output_names,
        )

        chunk_rows = token_ids.shape[0]
        row_end = row_start + chunk_rows
        valid = min(row_end, n_rows) - row_start  # drop padding rows past n_rows

        chunk_pos = row_pos[row_start : row_start + valid]
        # gather the swept position's d_model vector for each row in the chunk
        batch_arange = np.arange(valid)

        for i, output_name in enumerate(out_output_names[1:], start=1):
            layer_out = fx[i][:valid]  # (valid, cfg.seq, d_model)
            swept_vecs = layer_out[batch_arange, chunk_pos].to(torch.float32).detach().cpu().numpy()

            ds = datasets[output_name]
            ds["vec"][row_start : row_start + valid] = swept_vecs
            ds["sample_idx"][row_start : row_start + valid] = row_sample_idx[
                row_start : row_start + valid
            ]
            ds["pos"][row_start : row_start + valid] = chunk_pos
            ds["vocab_id"][row_start : row_start + valid] = row_vocab_id[
                row_start : row_start + valid
            ]

        row_start = row_end

        if (c + 1) % flush_every == 0 or c == n_chunks - 1:
            h5f.flush()

print(f"Saved perturbed sweep outputs to {perturbed_h5_path}")


100%|██████████| 25000/25000 [24:19<00:00, 17.12it/s] 


RuntimeError: Can't decrement id ref count (unable to close file, errno = 122, error message = 'Disk quota exceeded')

In [ ]:
import os

h5_path = "onnx_models/smoll/local_tangent_spaces.h5"
os.makedirs(os.path.dirname(h5_path), exist_ok=True)

# Build the manifold from the perturbed sweep saved to perturbed_h5_path above
# (not from the old `fxs`, which required the commented-out cell and is no
# longer populated). Each group in perturbed_sweep.h5 already holds one flat
# row per perturbed input: vec (n_rows, d_model), plus sample_idx / pos /
# vocab_id annotating which sample, swept sequence position, and substituted
# vocab id produced that row. perturbed_sweep.h5 also holds a top-level
# base_seqs (n_samples, cfg.seq) dataset -- the original sequence a row was
# derived from is base_seqs[sample_idx], so it's not duplicated per-row here.
# Flushed every flush_every queries so a crash/interrupt partway through
# still leaves a readable file with whatever rows made it in so far.
flush_every = 500  # queries

with h5py.File(perturbed_h5_path, "r") as sweep_h5f, h5py.File(h5_path, "w") as h5f:
    for output_name in out_output_names[1:]:
        sweep_grp = sweep_h5f[output_name]
        flat = torch.from_numpy(sweep_grp["vec"][:]).to(DEVICE)
        row_sample_idx = sweep_grp["sample_idx"][:]
        row_pos = sweep_grp["pos"][:]
        row_vocab_id = sweep_grp["vocab_id"][:]
        d_model = flat.shape[1]

        kns = knn_gpu(flat, k=cfg.d_model + 1)
        n_queries = kns[1].shape[0]

        grp = h5f.create_group(output_name)
        grp.attrs["output_name"] = output_name
        grp.attrs["d_model"] = d_model

        tangent_ds = grp.create_dataset(
            "tangent_basis", shape=(n_queries, d_model, d_model), dtype="float32"
        )
        k_ds = grp.create_dataset("k", shape=(n_queries,), dtype="int64")
        sample_idx_ds = grp.create_dataset("sample_idx", shape=(n_queries,), dtype="int64")
        pos_ds = grp.create_dataset("pos", shape=(n_queries,), dtype="int64")
        vocab_id_ds = grp.create_dataset("vocab_id", shape=(n_queries,), dtype="int64")
        neighbor_vocab_ids_ds = grp.create_dataset(
            "neighbor_vocab_ids", shape=(n_queries, cfg.d_model + 1), dtype="int64"
        )

        for j in tqdm.tqdm(range(n_queries), desc=output_name):
            neighbor_row_idx = kns[1][j]
            tangent_basis, k = estimate_local_tangent_space_p_var(
                flat[neighbor_row_idx], p=1
            )

            tangent_ds[j] = tangent_basis.detach().cpu().numpy()
            k_ds[j] = int(k)
            sample_idx_ds[j] = row_sample_idx[j]
            pos_ds[j] = row_pos[j]
            vocab_id_ds[j] = row_vocab_id[j]
            neighbor_row_idx_cpu = neighbor_row_idx.cpu().numpy()
            neighbor_vocab_ids_ds[j] = row_vocab_id[neighbor_row_idx_cpu]

            if (j + 1) % flush_every == 0 or j == n_queries - 1:
                h5f.flush()

print(f"Saved local tangent spaces to {h5_path}")
